# Practice 5-2. 지식 그래프 구축 실행 — `graphrag index`

`01-graphrag-lm-studio-setup.ipynb`에서 `working_directory`에 LM Studio용 설정(`settings.yaml`, `.env`)과 입력 문서를
준비해 두었습니다. 이 노트북은 실제로 `graphrag index`를 실행해 지식 그래프를 구축하고, 산출물을 확인합니다.

책의 설명대로 그래프 DB 구축은 크게 두 범주로 나뉩니다.

1. **Base processing 단계**: 문서 분할 → 엔티티 추출 → 기본 그래프 생성 → 커뮤니티 탐지 → 임베딩
2. **Enrichment(고도화) 단계**: 엔티티·관계·커뮤니티 데이터 정제, 커뮤니티 요약 생성

설치된 GraphRAG 3.x는 이를 다음 10개 워크플로우로 실행합니다(책이 설명하는 7단계와 이름은 다르지만 역할은
동일합니다): `load_input_documents → create_base_text_units → create_final_documents → extract_graph →
finalize_graph → extract_covariates → create_communities → create_final_text_units →
create_community_reports → generate_text_embeddings`.

> **로컬 LLM 소요 시간 참고**: 로컬 35B 모델로 엔티티 추출·커뮤니티 리포트를 생성하다 보니 요청당 평균
> 40~70초가 걸립니다. 이 책의 예제 문서(`How_to_invest_money.txt`, 약 150KB) 전체를 인덱싱하는 데는
> 수십 분 이상 소요될 수 있습니다.

## 0. 환경 설정

In [1]:
from pathlib import Path

WORKING_DIR = Path("working_directory")
assert (WORKING_DIR / "settings.yaml").exists(), "01-graphrag-lm-studio-setup.ipynb를 먼저 실행해 working_directory를 준비하세요"
print("작업 디렉터리:", WORKING_DIR.resolve())

작업 디렉터리: /workspace/day_05/working_directory


## 1. 인덱스 빌드 실행

`graphrag index --root ./working_directory`를 실행합니다. 워크플로우별 진행 상황이 실시간으로 출력됩니다.

In [2]:
!graphrag index --root ./working_directory

/opt/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
Starting pipeline with workflows: load_input_documents, create_base_text_units, create_final_documents, extract_graph, finalize_graph, extract_covariates, create_communities, create_final_text_units, create_community_reports, generate_text_embeddings
Starting workflow: load_input_documents

Workflow complete: load_input_documents
Starting workflow: create_base_text_units
  1 / 1 ............................................................................................
Workflow complete: create_base_text_units
Starting workflow: create_final_documents

Workflow complete: create_final_documents
Starting workflow: extract_graph
  294 / 294 ........................................................................................
Workflow complete: extract_graph
Starting workflow: finalize_graph

Workflow complete: finalize_graph
Starting workflow: ext

## 2. 산출물 확인

인덱싱이 끝나면 `working_directory/output`에 parquet 파일들이 생성됩니다. 각 파일이 책의 7단계 워크플로우
산출물에 대응합니다.

In [3]:
output_dir = WORKING_DIR / "output"
for p in sorted(output_dir.glob("*.parquet")):
    print(f"{p.name:35s} {p.stat().st_size / 1024:8.1f} KB")

communities.parquet                     17.0 KB
community_reports.parquet               97.6 KB
documents.parquet                       86.8 KB
entities.parquet                        30.9 KB
relationships.parquet                   29.8 KB
text_units.parquet                     101.1 KB


### 2.1 최종 엔티티 데이터 — `entities.parquet`

`extract_graph`(엔티티 추출 → 통합) 단계의 산출물입니다. 책의 `PROJECT GUTENBERG` 예시처럼, 각 엔티티는
설명(`description`)과 연결 정도(`degree`, 다른 엔티티와의 관계 수)를 갖습니다.

In [4]:
import pandas as pd

entities = pd.read_parquet(output_dir / "entities.parquet")
print("엔티티 수:", len(entities))
entities[["title", "type", "description", "degree"]].head(10)

엔티티 수: 157


,title,type,description,degree
0,PROJECT GUTENBERG EBOOK HOW TO INVEST MONEY,EVENT,An ebook detailing investment principles publi...,3
1,GEORGE GARR HENRY,PERSON,The author of the book How to Invest Money,1
2,JULIA NEUFELD,PERSON,One of the producers/proofreaders for the eBook,1
3,VICE-PRESIDENT GUARANTY TRUST COMPANY,ORGANIZATION,A financial institution mentioned in connectio...,0
4,FUNK & WAGNALLS COMPANY,ORGANIZATION,A company involved in printing/copyrighting th...,1
5,MONEY,ORGANIZATION,Money is discussed in the context of working c...,0
6,INVESTMENT BANKER,PERSON,An experienced and reliable professional consu...,0
7,BUSINESS MAN,PERSON,A BUSINESS MAN is characterized as an individu...,1
8,BANKER'S BUSINESS,ORGANIZATION,The investment of money is described as a bank...,0
9,WIDOWS AND ORPHANS' FUNDS,ORGANIZATION,Funds belonging to widows and orphans that are...,0


### 2.2 커뮤니티 구조 — `communities.parquet`

`create_communities` 단계(Leiden 클러스터링)의 산출물입니다. 이 GraphRAG 버전은 `settings.yaml`의
`snapshots.embeddings: false` 기본값 때문에 책이 설명하는 그래프 임베딩·2D 좌표(x, y)는 생성하지 않지만,
각 커뮤니티가 어떤 엔티티들을 묶는지(`entity_ids`)와 계층 구조(`level`, `parent`, `children`)는 그대로
확인할 수 있습니다.

In [5]:
communities_df = pd.read_parquet(output_dir / "communities.parquet")
sample = communities_df.iloc[0]
print({
    "community": sample["community"],
    "level": sample["level"],
    "title": sample["title"],
    "size(엔티티 수)": sample["size"],
    "children": list(sample["children"]),
})

{'community': np.int64(0), 'level': np.int64(0), 'title': 'Community 0', 'size(엔티티 수)': np.int64(8), 'children': []}


### 2.3 커뮤니티 요약문 — `community_reports.parquet`

`create_final_community_reports` 단계 산출물로, 각 커뮤니티에 대한 제목·요약·중요도(rating)·세부 발견
(findings)이 담겨 있습니다. 글로벌 검색(`03-graphrag-global-local-search.ipynb`)이 바로 이 리포트들을 맵-리듀스로 훑습니다.

In [6]:
community_reports = pd.read_parquet(output_dir / "community_reports.parquet")
print("커뮤니티 리포트 수:", len(community_reports))

sample = community_reports.iloc[0]
print({
    "community": sample["community"],
    "level": sample["level"],
    "rank": sample["rank"],
    "title": sample["title"],
    "summary": str(sample["summary"])[:200] + "...",
})

커뮤니티 리포트 수: 14
{'community': np.int64(8), 'level': np.int64(1), 'rank': np.float64(4.0), 'title': 'Mortgage Bonds and Receiver Relationship', 'summary': 'This community centers around the relationship between Mortgage Bonds and a Receiver, which is appointed when a railroad cannot meet its interest charges. The core dynamic involves mortgage bondholder...'}


### 2.4 전체 산출물 요약

In [7]:
relationships = pd.read_parquet(output_dir / "relationships.parquet")
communities = pd.read_parquet(output_dir / "communities.parquet")
text_units = pd.read_parquet(output_dir / "text_units.parquet")

print(f"텍스트 유닛(청크) 수 : {len(text_units)}")
print(f"엔티티 수            : {len(entities)}")
print(f"관계 수              : {len(relationships)}")
print(f"커뮤니티 수          : {len(communities)}")
print(f"커뮤니티 리포트 수   : {len(community_reports)}")

텍스트 유닛(청크) 수 : 29
엔티티 수            : 157
관계 수              : 131
커뮤니티 수          : 14
커뮤니티 리포트 수   : 14


그래프 DB 구축이 끝났습니다. 산출물은 `working_directory/output`에 parquet 파일로 저장되어 있고,
임베딩은 `output/lancedb`에 로컬 벡터 인덱스(LanceDB)로 저장되어 있습니다. 구글 코랩과 달리 이 워크스페이스는
런타임이 끝나도 파일이 유지되므로 별도로 구글 드라이브에 백업할 필요가 없습니다.

다음 노트북(`03-graphrag-global-local-search.ipynb`)에서 이 그래프 DB를 대상으로 로컬 검색과 글로벌 검색을 실습합니다.